# Phase 10 -- Refined thematic analysis pipeline

End-to-end pipeline that classifies free-text consultation responses against
four predefined themes, extracts evidence quotes, validates them, and
discovers sub-themes within each theme. Overall-stance content (overall
support / partial support / opposition) is folded into the relevant content
theme's definition rather than being its own theme.

Pipeline shape:

1. **Stage 1** -- per-theme Yes/No classification for the four themes.
2. **Stage 2** -- ask the LLM for up to 3 candidate verbatim quotes per Yes
   pair, each with its own reasoning. Empty list allowed. No ranking.
3. **Stage 3** -- per-quote validation (one LLM call per candidate). Every
   Yes is kept; failures stay in the audit.
4. **Stage 4** -- pairwise overlap calculation between themes on the same
   response. No automatic resolution; the result lives in an `Overlaps With
   Themes` column for the human reviewer to choose how to handle.
5. **Stage 5a + 5b** -- sub-theme discovery per parent theme from the full
   validated set; per-quote sub-theme assignment.

The human reading the audit / report picks which quotes stand out -- the
pipeline does not curate "best quotes". Output is three CSVs plus a
stakeholder-facing report:

- `phase10_pipeline_audit.csv` -- end-to-end audit, one row per
  (response x theme x candidate). Every decision and its reasoning in one
  place.
- `phase10_assurance_per_theme_metrics.csv` -- per-theme accuracy /
  precision / recall + survival counts.
- `phase10_assurance_run_metadata.csv` -- single-row run provenance.
- `phase10_report.md` / `phase10_report.html` -- readable summary with
  top exemplars and sub-themes per theme.

Read top to bottom; each stage is one cell with the prompt + loop + LLM
call inlined.


## Setup and data load

Load the responses sheet (`Q1&2`) from the Tagger workbook and the theme
definitions from `Definitions.xlsx`. Tidy unicode artefacts in the free text,
build `themes_dict` (name -> definition), wire up the Azure OpenAI client,
and pin runtime config (`MODEL_DEPLOYMENT = "gpt-4o"`, `TEMPERATURE = 0.0`,
`PAUSE_SECONDS = 0.5`).


In [ ]:
import pandas as pd

In [ ]:
themes = pd.read_excel("Definitions.xlsx", sheet_name="Sheet1")
df = pd.read_excel("Tagger RSR Experimentation Dataset.xlsx", sheet_name="Q1&2")


In [ ]:
# Remove all unicode artifacts.

import unicodedata
import re

def clean_text(x):
    if not isinstance(x, str):
        return x
    x = unicodedata.normalize("NFKC", x)          # normalize unicode
    x = re.sub(r"\s+", " ", x)                    # collapse all whitespace
    return x.strip()

themes = themes.map(clean_text)
df["Free Words"] = df["Free Words"].map(clean_text)

In [ ]:
themes_dict = dict(zip(themes.iloc[:, 1], themes.iloc[:, 2]))

In [ ]:
import os
import time
from openai import AzureOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal, Union

In [ ]:
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4.1-nano")
AZURE_OPENAI_EMBED_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBED_DEPLOYMENT")

In [ ]:
client = AzureOpenAI(api_key=os.getenv("AZURE_OPENAI_KEY"), azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"), api_version="2024-02-01")

In [ ]:
TEMPERATURE = 0.0
PAUSE_SECONDS = 0.5
MODEL_DEPLOYMENT = "gpt-4o"

## Stage 1 -- Per-theme Yes/No classification

For every (response x theme) pair, ask the LLM whether the free text
evidences the theme. Returns a structured verdict + reasoning + confidence.


In [ ]:
class ThemeClassification(BaseModel):
    reasoning: str = Field(description="Short summation (1-2 sentences) of why the statement does or does not evidence the theme.")
    verdict: Literal["Yes", "No"] = Field(description="Yes if the statement clearly evidences the theme, otherwise No.")
    confidence: float = Field(description="Percentage confidence in the verdict, as a number from 0 to 100 and without the percentage sign (%).")

In [ ]:
THEME_PROMPT_TEMPLATE = """ You are classifying consultation free-text against a predefined theme.

Context: 
The responses are feedback on proposed changes to charges that the Environment Agency plans to levy. 

Theme name:
{theme_name}

Theme definition: 
{theme_definition}

Statement:
{free_text}

Tasks:
Decide whether the statement provides clear evidence of the theme. Reason through the decision before answering. 

Classification Principle:
Classify against the theme name, clarifying against the theme definition to ensure that the response meets the criteria defined within the theme.

Rules:
- Yes -- only where the statement clearly describes or gives evidence of the theme as defined.
- Yes -- only if the link to the theme is present in the wording itself, not inferred.
- No -- where the statement is too vague, generic, or would require assumptions.
- No -- where the statement could fit the theme only because of wider context rather than the wording of the statement. 
- Base the decision only on the statement. 

Reasoning steps (work through these before producing your answer):
1. Identify what the statement actually says. 
2. Compare this against the theme definition. 
3. Decide whether the link is explicit in the statement itself. 
4. Summarise your reasoning in 1-2 sentences.
5. Assign a percentage confidence score from 0 to 100 reflecting how certain you are, without the percentage sign (%)
"""

In [ ]:
def classify_theme(df, theme_name, theme_definition, client, text_col="Free Words"):
    verdict_col = f"{theme_name} - AI Verdict"
    reasoning_col = f"{theme_name} - AI Reasoning"
    confidence_col = f"{theme_name} - AI Confidence"

    def classify_one(free_text):
        if pd.isna(free_text) or str(free_text).strip() == "":
            return None

        prompt = THEME_PROMPT_TEMPLATE.format(theme_name=theme_name, theme_definition=theme_definition, free_text=free_text)

        try:
            response = client.beta.chat.completions.parse(model=MODEL_DEPLOYMENT, messages=[{"role": "user", "content": prompt}], temperature=TEMPERATURE, response_format=ThemeClassification)
            time.sleep(PAUSE_SECONDS)
            return response.choices[0].message.parsed

        except Exception as e:
            return ThemeClassification(reasoning=f"ERROR: {str(e)[:150]}", verdict="No", confidence=0.0)

    results = df[text_col].apply(classify_one)
    new_cols = pd.DataFrame({verdict_col: results.apply(lambda r: r.verdict if r else None), reasoning_col: results.apply(lambda r: r.reasoning if r else None), confidence_col: results.apply(lambda r: r.confidence if r else None),}, index=df.index)
    df = pd.concat([df, new_cols], axis=1)

    return df

In [ ]:
for theme_name, theme_definition in themes_dict.items():
    df = classify_theme(df, theme_name, theme_definition, client)


## Shared helpers used by Stage 2 onwards

Small utilities that the pipeline below leans on:

- **`quote_in_text`** -- tolerant Ctrl-F-style verbatim check that lets the
  LLM's cosmetic edits (capitalisation, trailing full stop) pass while still
  rejecting fabricated wording.
- **`_normalise_for_overlap`, `_overlap_ratio`** -- longest-common-substring
  ratio between two quotes (used for the overlap-flag column).
- **Sub-theme primitives** -- the Pydantic models and prompt templates
  the sub-theme discovery / consolidation calls use.

You can read these in detail if you want to understand the internals; for
following the pipeline flow it's enough to know they exist.


In [ ]:
# --- Verbatim check ----------------------------------------------------------
import string
from difflib import SequenceMatcher
import random
from pydantic import create_model

_BOUNDARY_TRIM = string.punctuation + string.whitespace + "‘’“”–—…"


def quote_in_text(quote, free_text):
    """Tolerant substring check used after the LLM proposes a quote.

    Lowercases both sides, collapses whitespace, strips boundary punctuation,
    so cosmetic edits (capitalising the first letter, adding a trailing full
    stop, wrapping in quotation marks) don't trigger a false negative. The
    free text is left intact apart from outer whitespace.
    """
    if quote is None or free_text is None:
        return False
    q = re.sub(r"\s+", " ", str(quote)).strip(_BOUNDARY_TRIM).lower()
    t = re.sub(r"\s+", " ", str(free_text)).strip().lower()
    if not q:
        return False
    return q in t


# --- Overlap helpers ---------------------------------------------------------
def _normalise_for_overlap(quote):
    if pd.isna(quote):
        return ""
    return re.sub(r"\s+", " ", str(quote)).strip(_BOUNDARY_TRIM).lower()


def _longest_match(a, b):
    if not a or not b:
        return 0, ""
    m = SequenceMatcher(None, a, b, autojunk=False).find_longest_match(0, len(a), 0, len(b))
    return m.size, a[m.a:m.a + m.size]


def _overlap_ratio(q_a, q_b):
    """Longest common substring length divided by the length of the shorter quote."""
    if not q_a or not q_b:
        return 0.0, ""
    size, shared = _longest_match(q_a, q_b)
    return size / min(len(q_a), len(q_b)), shared


# --- Sub-theme primitives ----------------------------------------------------
from typing import List as _List

BATCH_SIZE = 20
SHUFFLE_SEED = 42
NONE_LABEL = "None"


class CandidateSubTheme(BaseModel):
    name: str = Field(description="Short name (2-5 words) for the proposed sub-theme.")
    definition: str = Field(description="One-line definition of what the sub-theme captures.")
    reasoning: str = Field(description="1-2 sentences on why this grouping is distinct from the others in this batch.")
    quote_indices: _List[int] = Field(description="Indices (from the batch shown in the prompt) of quotes that support this sub-theme.")


class SubThemeBatch(BaseModel):
    sub_themes: _List[CandidateSubTheme] = Field(description="3-6 candidate sub-themes covering this batch of quotes.")


class ConsolidatedSubTheme(BaseModel):
    name: str = Field(description="Short name (2-5 words) for the final sub-theme.")
    definition: str = Field(description="One-line definition of what the sub-theme captures.")
    merged_from: _List[str] = Field(description="Names of the candidate sub-themes that were merged into this one.")


class ConsolidatedSubThemes(BaseModel):
    reasoning: str = Field(description="1-3 sentences on how the candidates were grouped and what was dropped or refined.")
    sub_themes: _List[ConsolidatedSubTheme] = Field(description="3-8 consolidated sub-themes covering the parent theme.")


PROPOSE_PROMPT_TEMPLATE = """You are identifying sub-themes within a single predefined theme from consultation free-text.

Context:
The responses are feedback on proposed changes to charges that the Environment Agency plans to levy. All the quotes below have already been classified as evidence of the same parent theme. Your job is to read them and propose 3-6 finer-grained sub-themes describing the patterns you see.

Parent theme name: {theme_name}
Parent theme definition: {theme_definition}

Quotes (indexed):
{quote_block}

Rules:
- Propose between 3 and 6 sub-themes.
- Each name should be short (2-5 words).
- Each definition should be one line.
- Sub-themes should be substantively different from each other.
- For each sub-theme, list the indices of the quotes above that support it. A quote may appear under more than one sub-theme if it genuinely fits both.
- Base the sub-themes on the wording of the quotes, not on assumptions.
"""


CONSOLIDATE_PROMPT_TEMPLATE = """You are consolidating candidate sub-themes for a single predefined theme.

Context:
The candidate sub-themes below were each proposed from a different batch of quotes belonging to the same parent theme. Many will overlap or describe the same idea in different words.

Parent theme name: {theme_name}
Parent theme definition: {theme_definition}

Candidate sub-themes:
{candidate_block}

Task:
Produce a final list of 3-8 consolidated sub-themes that together cover the parent theme.

Rules:
- Group near-equivalent candidates under one consolidated sub-theme.
- Drop candidates that overlap fully with another.
- Refine the wording so each definition is clear and the set as a whole is distinct.
- Each consolidated sub-theme should be substantively different from the others.
- Name each one in 2-5 words.
- For each consolidated sub-theme, list the names of the original candidates it absorbs.
"""


def propose_sub_themes_for_batch(theme_name, theme_definition, quotes_batch):
    """Ask the LLM for candidate sub-themes from one batch of quotes."""
    quote_block = "\n".join(f"[{i}] {q}" for i, q in enumerate(quotes_batch))
    prompt = PROPOSE_PROMPT_TEMPLATE.format(
        theme_name=theme_name,
        theme_definition=theme_definition,
        quote_block=quote_block,
    )
    response = client.beta.chat.completions.parse(
        model=MODEL_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        response_format=SubThemeBatch,
    )
    time.sleep(PAUSE_SECONDS)
    return response.choices[0].message.parsed.sub_themes


def consolidate_sub_themes(theme_name, theme_definition, candidate_sub_themes):
    """Merge all batch-level candidates into a single consolidated sub-theme list."""
    candidate_block = "\n".join(f"- {c.name}: {c.definition}" for c in candidate_sub_themes)
    prompt = CONSOLIDATE_PROMPT_TEMPLATE.format(
        theme_name=theme_name,
        theme_definition=theme_definition,
        candidate_block=candidate_block,
    )
    response = client.beta.chat.completions.parse(
        model=MODEL_DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        response_format=ConsolidatedSubThemes,
    )
    time.sleep(PAUSE_SECONDS)
    return response.choices[0].message.parsed


## Pipeline (Stage 2 onwards)

Builds on `df` (Stages 1 + 1b already populated) and writes the three output
CSVs at the end. Each stage is one code cell: prompt template at the top,
loop with the LLM call inlined below.


In [ ]:
# Pipeline setup: knobs, working dataframe, and all the structured-output shapes
# the LLM calls below use. After this cell:
#   * df already carries the verdict/reasoning/confidence columns Stages 1 + 1b wrote.
#   * the Pydantic classes are available for `response_format=` in every LLM call.

# Knobs.
TOP_QUOTES_K = 3            # Stage 2: max candidate quotes per pair.
OVERLAP_THRESHOLD = 0.7     # overlap ratio above which two themes get flagged.

# Stage 3 and Stage 5b ask a focused per-quote question (verdict / sub-theme).
# A smaller model is more than capable of these and dramatically cheaper.
# Stages 1, 2, 5a stay on MODEL_DEPLOYMENT (the larger model from the config
# cell above).
MODEL_VALIDATE = "gpt-4o-mini"


# Structured output shapes -- one Pydantic class per LLM call's return type.

class CandidateQuote(BaseModel):
    quote: str = Field(description="A verbatim quote, character-for-character from the Free Text.")
    reasoning: str = Field(description="1-2 sentences explaining why this quote evidences the theme.")

class CandidateQuotes(BaseModel):
    quotes: List[CandidateQuote] = Field(description="Up to 3 distinct verbatim quotes that evidence the theme. Empty list if no honest quote exists. No need to rank -- list whatever fits.")


class QuoteValidation(BaseModel):
    verdict: Literal["Yes", "No"] = Field(description="Yes if the quote clearly evidences the theme as defined.")
    reasoning: str = Field(description="1-2 sentences explaining the verdict.")


class SubThemeAssignment(BaseModel):
    chosen_sub_theme: str = Field(description="The sub-theme the quote best fits, or 'None'.")
    reasoning: str = Field(description="1-2 sentences explaining the assignment.")
    confidence: float = Field(description="Percentage confidence 0-100, no % sign.")


### Stage 2 -- candidate quotes per Yes pair

Ask the LLM for up to 5 distinct verbatim quotes per (response × theme) that
got a Yes in Stage 1. Each quote comes with its own one-sentence reasoning.
Empty list allowed when nothing honestly fits. No ranking required -- just
distinct quotes. Builds `candidates_df` -- one row per (response × theme ×
candidate quote).


In [ ]:
TOP_K_QUOTES_PROMPT = """You are extracting candidate evidence quotes for a theme from consultation free-text.

Theme name: {theme_name}

Theme definition:
{theme_definition}

Free Text:
{free_text}

Task: Identify up to {top_k} verbatim quotes from the Free Text that evidence the theme. Each quote must:

- Be a character-for-character copy of a contiguous span of the Free Text.
- Be distinct from the others (no near-duplicate wording).
- Be accompanied by 1-2 sentences of reasoning.

You do NOT need to rank the quotes. List up to {top_k} distinct quotes that honestly evidence the theme. Return fewer than {top_k} if fewer fit. Return an empty list if no contiguous span would evidence the theme.

VERBATIM RULES -- each quote MUST be character-for-character:
- Do NOT paraphrase, summarise, or rewrite.
- Do NOT change capitalisation, fix typos, or alter punctuation.
- Do NOT wrap in quotation marks or brackets.

An empty list is always preferable to fabricated quotes.
"""


# Loop -- for every Yes pair, ask the LLM for candidate quotes and record each
# one. candidates_df accumulates as we go. No rank / ordering -- just the
# quote, its extraction reasoning, and whether it survived the verbatim check.
candidate_records = []

for theme_name, theme_definition in themes_dict.items():
    verdict_col = f"{theme_name} - AI Verdict"
    for idx, row in df.iterrows():
        if row.get(verdict_col) != "Yes":
            continue
        free_text = row["Free Words"]
        if pd.isna(free_text) or str(free_text).strip() == "":
            continue

        prompt = TOP_K_QUOTES_PROMPT.format(
            theme_name=theme_name,
            theme_definition=theme_definition,
            free_text=free_text,
            top_k=TOP_QUOTES_K,
        )

        try:
            response = client.beta.chat.completions.parse(
                model=MODEL_DEPLOYMENT,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                response_format=CandidateQuotes,
            )
            time.sleep(PAUSE_SECONDS)
            quotes = response.choices[0].message.parsed.quotes
        except Exception as e:
            # API error -- record a sentinel row and move on. quote_in_text
            # is left None so this row doesn't count as a real candidate
            # in the metrics.
            candidate_records.append({
                "row_idx": idx,
                "Parent theme": theme_name,
                "quote": None,
                "extraction_reasoning": f"ERROR: {str(e)[:150]}",
                "quote_in_text": None,
            })
            continue

        if not quotes:
            # LLM said: no honest verbatim quote exists. Sentinel row.
            candidate_records.append({
                "row_idx": idx,
                "Parent theme": theme_name,
                "quote": None,
                "extraction_reasoning": "LLM returned empty list -- no honest verbatim quote in the free text.",
                "quote_in_text": None,
            })
            continue

        # One row per real candidate. Quote nulled if verbatim check fails;
        # quote_in_text is set to True or False (never None) so this row
        # counts as a real candidate in the metrics.
        for cand in quotes:
            in_text = quote_in_text(cand.quote, free_text)
            candidate_records.append({
                "row_idx": idx,
                "Parent theme": theme_name,
                "quote": cand.quote if in_text else None,
                "extraction_reasoning": cand.reasoning,
                "quote_in_text": in_text,
            })

candidates_df = pd.DataFrame(candidate_records)
candidates_df.head()


### Stage 3 -- per-quote validation

For every candidate quote, send a single LLM call: theme definition + this
one quote, and a yes/no on whether the quote evidences the theme. No
batching, no candidate_rank, no swap risk -- the LLM only sees one quote at
a time, so there's nothing to identify.

Uses a smaller cheaper model (`MODEL_VALIDATE`) because this is a focused
narrow question and the per-call cost matters when we make many calls.


In [ ]:
VALIDATE_PROMPT = """You are checking whether a candidate quote evidences a theme in consultation free-text classification.

Theme name: {theme_name}

Theme definition:
{theme_definition}

Quote: "{quote}"

Task: Decide whether the quote clearly evidences the theme as defined.

Rules:
- Yes -- only where the quote clearly evidences the theme as defined.
- Yes -- only if the link is in the wording itself, not inferred.
- No -- where the quote is too vague, generic, or would need assumptions.
- Base the decision only on the quote itself.
"""


candidates_df["validation_verdict"] = None
candidates_df["validation_reasoning"] = None

# One LLM call per candidate quote -- nothing to swap, nothing to identify.
mask = candidates_df["quote"].notna()
for i in candidates_df[mask].index:
    theme_name = candidates_df.at[i, "Parent theme"]
    quote = candidates_df.at[i, "quote"]
    prompt = VALIDATE_PROMPT.format(
        theme_name=theme_name,
        theme_definition=themes_dict[theme_name],
        quote=quote,
    )
    try:
        response = client.beta.chat.completions.parse(
            model=MODEL_VALIDATE,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            response_format=QuoteValidation,
        )
        time.sleep(PAUSE_SECONDS)
        v = response.choices[0].message.parsed
        candidates_df.at[i, "validation_verdict"] = v.verdict
        candidates_df.at[i, "validation_reasoning"] = v.reasoning
    except Exception as e:
        candidates_df.at[i, "validation_verdict"] = "ERROR"
        candidates_df.at[i, "validation_reasoning"] = f"ERROR: {str(e)[:150]}"


candidates_df.head()


### Stage 4 -- overlap calculation (column-only, no resolution)

For each (response × theme) pair, compare its kept-for-downstream quotes
against every other theme's kept quotes on the same response. Any cross-theme
pair with longest-common-substring ratio above `OVERLAP_THRESHOLD` (0.7)
is flagged in an `overlaps_with_themes` column. The pipeline never picks a
winner; the column is just for human reviewers to see and decide.


In [ ]:
# For each response, compare every theme-with-validated-quotes against every
# other theme-with-validated-quotes. Pairs whose strongest overlap >= the
# threshold are flagged in overlaps_with_themes -- a human-readable list of
# the other themes (and ratios) that share substantial wording with this one.
# Pure post-processing: no LLM call, just substring math on the existing
# candidates_df.

candidates_df["overlaps_with_themes"] = None

# Only validated quotes can overlap with anything. Group by response so we
# can pairwise-compare themes within a single response.
validated = candidates_df[
    (candidates_df["validation_verdict"] == "Yes") & candidates_df["quote"].notna()
]
overlaps_by_pair = {}  # (row_idx, theme_name) -> ["OtherTheme (0.91)", ...]

for row_idx, row_group in validated.groupby("row_idx"):
    # Collect normalised quotes per theme on this response.
    themes_on_row = []
    for theme_name, theme_group in row_group.groupby("Parent theme"):
        normalised = [_normalise_for_overlap(q) for q in theme_group["quote"]]
        normalised = [n for n in normalised if n]
        if normalised:
            themes_on_row.append((theme_name, normalised))

    # Pairwise compare every two themes' quotes; take the max ratio across
    # all quote-pairs between them. If above threshold, record on both sides.
    for i in range(len(themes_on_row)):
        for j in range(i + 1, len(themes_on_row)):
            name_a, qs_a = themes_on_row[i]
            name_b, qs_b = themes_on_row[j]
            max_ratio = 0.0
            for q_a in qs_a:
                for q_b in qs_b:
                    ratio, _ = _overlap_ratio(q_a, q_b)
                    if ratio > max_ratio:
                        max_ratio = ratio
            if max_ratio >= OVERLAP_THRESHOLD:
                overlaps_by_pair.setdefault((row_idx, name_a), []).append(f"{name_b} ({max_ratio:.2f})")
                overlaps_by_pair.setdefault((row_idx, name_b), []).append(f"{name_a} ({max_ratio:.2f})")

# Write the overlap list back as a column on every candidate row of the pair.
for (row_idx, theme_name), overlaps in overlaps_by_pair.items():
    mask = (candidates_df["row_idx"] == row_idx) & (candidates_df["Parent theme"] == theme_name)
    candidates_df.loc[mask, "overlaps_with_themes"] = ", ".join(overlaps)


### Stage 5a -- sub-theme discovery per parent theme

For each parent theme, gather all validated quotes (across responses),
shuffle (with a fixed seed), and split into batches of 20. Each batch goes
to the AI for 3-6 candidate sub-theme proposals (name + one-line definition
each). Candidates from all batches are then consolidated by one final AI
call into 3-8 overarching sub-themes per parent theme.


In [ ]:
sub_themes_by_theme = {}
candidates_by_theme = {}

for theme_name, theme_definition in themes_dict.items():
    # Gather every kept quote for this theme.
    mask = (
        (candidates_df["Parent theme"] == theme_name)
        & (candidates_df["validation_verdict"] == "Yes")
        & candidates_df["quote"].notna()
    )
    quotes = candidates_df[mask]["quote"].tolist()
    if not quotes:
        continue

    # Shuffle (reproducible) and batch.
    random.seed(SHUFFLE_SEED)
    shuffled = list(quotes)
    random.shuffle(shuffled)

    all_candidates = []
    for start in range(0, len(shuffled), BATCH_SIZE):
        batch = shuffled[start:start + BATCH_SIZE]
        all_candidates.extend(propose_sub_themes_for_batch(theme_name, theme_definition, batch))

    candidates_by_theme[theme_name] = all_candidates
    if not all_candidates:
        continue

    # Consolidate the batch-level candidates into a final list of sub-themes.
    sub_themes_by_theme[theme_name] = consolidate_sub_themes(theme_name, theme_definition, all_candidates)

sub_themes_by_theme


### Stage 5b -- per-quote sub-theme assignment

For every validated quote, send a single LLM call: parent theme + sub-theme
list + this one quote, returning the chosen sub-theme (or `None`). No
batching, no rank fields, no swap risk -- the LLM only sees one quote per
call.

Uses `MODEL_VALIDATE` (the smaller, cheaper model) -- single-quote
multiple-choice is well within its capability.


In [ ]:
SUBTHEME_PROMPT = """You are assigning a single quote to a sub-theme within a parent theme.

Parent theme name: {theme_name}
Parent theme definition:
{theme_definition}

Sub-themes (choose exactly one, or "{none_label}" if no sub-theme fits):
{sub_theme_block}

Quote: "{quote}"

Task: Pick the sub-theme this quote best fits, or "{none_label}" if no sub-theme genuinely fits. Provide 1-2 sentences of reasoning and a percentage confidence (0-100, no % sign).

Rules:
- Copy the chosen sub-theme name exactly as written in the sub-themes list.
- "{none_label}" is always preferable to a weak fit.
"""


candidates_df["sub_theme"] = None
candidates_df["sub_theme_reasoning"] = None
candidates_df["sub_theme_confidence"] = None

for theme_name, consolidated in sub_themes_by_theme.items():
    sub_theme_block = "\n".join(f"- {s.name}: {s.definition}" for s in consolidated.sub_themes)
    sub_theme_block += f"\n- {NONE_LABEL}: no sub-theme genuinely fits this quote"

    # Allowed sub-theme names for this parent theme. Anything outside this
    # set is treated as NONE_LABEL -- we do not let the LLM invent
    # sub-theme names that aren't in the consolidated list.
    allowed_sub_themes = {s.name for s in consolidated.sub_themes} | {NONE_LABEL}

    mask = (
        (candidates_df["Parent theme"] == theme_name)
        & (candidates_df["validation_verdict"] == "Yes")
        & candidates_df["quote"].notna()
    )

    for i in candidates_df[mask].index:
        quote = candidates_df.at[i, "quote"]
        prompt = SUBTHEME_PROMPT.format(
            theme_name=theme_name,
            theme_definition=themes_dict[theme_name],
            none_label=NONE_LABEL,
            sub_theme_block=sub_theme_block,
            quote=quote,
        )
        try:
            response = client.beta.chat.completions.parse(
                model=MODEL_VALIDATE,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                response_format=SubThemeAssignment,
            )
            time.sleep(PAUSE_SECONDS)
            a = response.choices[0].message.parsed
        except Exception:
            continue

        chosen = a.chosen_sub_theme if a.chosen_sub_theme in allowed_sub_themes else NONE_LABEL
        candidates_df.at[i, "sub_theme"] = chosen
        candidates_df.at[i, "sub_theme_reasoning"] = a.reasoning
        candidates_df.at[i, "sub_theme_confidence"] = a.confidence


### Build the big audit CSV

One row per (response × theme × candidate quote), plus one "no candidates"
row for pairs where Stage 1 said No or Stage 2 produced nothing. All
decisions and reasoning side-by-side. The human label lookup and outcome
classification are inlined into the loop so you can read the whole audit
build top-to-bottom.


In [ ]:
# Theme -> column-name-in-input-sheet mapping for the human gold standard.
HUMAN_COLUMN_BY_THEME = {
    "How charge affects customers": "How charge affects customers",
    "Charge scheme approach": "Charge scheme approach",
    "Service provided by EA": "Service we provide",
    "Issue with consultation": "Issue with consultation",
}

# Re-read the input sheet so the human labels are not affected by anything
# the pipeline added to df. Missing file -> labels become "Unknown".
try:
    _human_df = pd.read_excel("Tagger RSR Experimentation Dataset.xlsx", sheet_name="Q1&2")
except Exception:
    _human_df = pd.DataFrame()


# Build audit_rows by straight loop. Per (response x theme), we look up the
# Stage 1 verdict, the human label, the strict-AI-Yes flag (Stage 1 Yes AND
# at least one survivor), and the confusion-matrix outcome -- then emit one
# row per candidate (or a single "no candidates" row). Per-pair candidates
# are enumerated locally just to build a unique pair_id; that number has no
# ranking meaning.
audit_rows = []

for theme_name in themes_dict:
    verdict_col = f"{theme_name} - AI Verdict"
    reason_col = f"{theme_name} - AI Reasoning"
    conf_col = f"{theme_name} - AI Confidence"
    human_col = HUMAN_COLUMN_BY_THEME.get(theme_name)

    for idx, row in df.iterrows():
        # Stage 1 outputs.
        stage1_verdict = row.get(verdict_col)
        stage1_reasoning = row.get(reason_col)
        stage1_confidence = row.get(conf_col)

        # Human label lookup.
        if human_col is None or _human_df.empty or human_col not in _human_df.columns:
            human_label = "Unknown"
        else:
            raw = _human_df.at[idx, human_col]
            human_label = "Yes" if str(raw).strip().upper() == "Y" else "No"

        # Strict AI Yes: Stage 1 said Yes AND at least one candidate survived validation.
        pair_candidates = candidates_df[
            (candidates_df["row_idx"] == idx) & (candidates_df["Parent theme"] == theme_name)
        ]
        ai_yes_strict = bool((pair_candidates["validation_verdict"] == "Yes").any()) if not pair_candidates.empty else False

        # Confusion-matrix outcome.
        if human_label == "Unknown":
            outcome = "Excluded (no human label)"
        elif human_label == "Yes" and ai_yes_strict:
            outcome = "True Positive"
        elif human_label == "Yes" and not ai_yes_strict:
            outcome = "False Negative"
        elif human_label == "No" and ai_yes_strict:
            outcome = "False Positive"
        else:
            outcome = "True Negative"

        # Per-pair base row -- everything except the candidate-specific fields.
        base = {
            "Citizen": row.get("Citizen "),
            "Free Words": row.get("Free Words"),
            "Parent theme": theme_name,
            "Stage 1 Verdict": stage1_verdict,
            "Stage 1 Reasoning": stage1_reasoning,
            "Stage 1 Confidence": stage1_confidence,
            "AI Yes (strict)": ai_yes_strict,
            "Human Label": human_label,
            "Outcome": outcome,
        }

        if pair_candidates.empty:
            # No candidates: emit one row with the pair-level info only.
            audit_rows.append({
                "pair_id": f"{idx}::{theme_name}",
                **base,
                "Quote Text": None,
                "Quote Extraction Reasoning": None,
                "Quote In Free Text": None,
                "Validation Verdict": None,
                "Validation Reasoning": None,
                "Sub-theme": None,
                "Sub-theme Reasoning": None,
                "Sub-theme Confidence": None,
                "Overlaps With Themes": None,
            })
            continue

        # One row per candidate. Local enumeration -> a unique pair_id; the
        # number has no quality meaning.
        for n, (_, cand) in enumerate(pair_candidates.iterrows(), start=1):
            audit_rows.append({
                "pair_id": f"{idx}::{theme_name}::{n}",
                **base,
                "Quote Text": cand["quote"],
                "Quote Extraction Reasoning": cand["extraction_reasoning"],
                "Quote In Free Text": cand["quote_in_text"],
                "Validation Verdict": cand["validation_verdict"],
                "Validation Reasoning": cand["validation_reasoning"],
                "Sub-theme": cand["sub_theme"],
                "Sub-theme Reasoning": cand["sub_theme_reasoning"],
                "Sub-theme Confidence": cand["sub_theme_confidence"],
                "Overlaps With Themes": cand["overlaps_with_themes"],
            })

audit_df = pd.DataFrame(audit_rows)
audit_df.head()


### Per-theme metrics

Per-theme accuracy / precision / recall + survival counts. Computed from
`audit_df` by dedup'ing to one row per pair (each pair contributes once
regardless of how many quotes it has) and counting outcomes per theme.


In [ ]:
# Dedup to one row per (row_idx, theme) for confusion-matrix counts.
pair_key = audit_df["pair_id"].str.split("::").str[:2].str.join("::")
pair_view = audit_df.assign(_pair_key=pair_key).drop_duplicates(subset="_pair_key")

metrics_records = []
for theme_name in themes_dict:
    sub = pair_view[pair_view["Parent theme"] == theme_name]
    tp = int((sub["Outcome"] == "True Positive").sum())
    tn = int((sub["Outcome"] == "True Negative").sum())
    fp = int((sub["Outcome"] == "False Positive").sum())
    fn = int((sub["Outcome"] == "False Negative").sum())
    n_scored = tp + tn + fp + fn

    accuracy = (tp + tn) / n_scored if n_scored else None
    precision = tp / (tp + fp) if (tp + fp) else None
    recall = tp / (tp + fn) if (tp + fn) else None

    theme_candidates = candidates_df[candidates_df["Parent theme"] == theme_name]
    metrics_records.append({
        "Parent theme": theme_name,
        "Has human-coded column": HUMAN_COLUMN_BY_THEME.get(theme_name) is not None,
        "n_strict_AI_Yes": int(sub["AI Yes (strict)"].sum()),
        "n_candidates_generated": int(theme_candidates["quote_in_text"].notna().sum()),
        "n_validation_survivors": int((theme_candidates["validation_verdict"] == "Yes").sum()),
        "n_pairs_with_overlap_flag": int(
            theme_candidates.dropna(subset=["overlaps_with_themes"])
            .drop_duplicates(subset=["row_idx", "Parent theme"]).shape[0]
        ),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "accuracy_vs_human": accuracy,
        "precision_vs_human": precision,
        "recall_vs_human": recall,
    })

metrics_df = pd.DataFrame(metrics_records)
metrics_df


### Save the three pipeline outputs

`pipeline_audit.csv` (everything), `phase10_assurance_per_theme_metrics.csv`
(per-theme accuracy/precision/recall + survival counts),
`phase10_assurance_run_metadata.csv` (single-row run provenance).


In [ ]:
# Pipeline outputs.
# Only three files: the big audit CSV (everything decision-related) plus two
# assurance files (per-theme metrics + run provenance).
audit_df.to_csv("phase10_pipeline_audit.csv", index=False)
metrics_df.to_csv("phase10_assurance_per_theme_metrics.csv", index=False)

from datetime import datetime, timezone

_overlap_pair_count = int(
    candidates_df.dropna(subset=["overlaps_with_themes"])
    .drop_duplicates(subset=["row_idx", "Parent theme"])
    .shape[0]
)

run_metadata_df = pd.DataFrame([{
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "Phase 10.ipynb",
    "pipeline_version": "phase10",
    "model_deployment": MODEL_DEPLOYMENT,
    "model_validate": MODEL_VALIDATE,
    "temperature": TEMPERATURE,
    "pause_seconds": PAUSE_SECONDS,
    "top_quotes_k": TOP_QUOTES_K,
    "overlap_threshold": OVERLAP_THRESHOLD,
    "n_responses": int(len(df)),
    "n_themes": int(len(themes_dict)),
    "n_audit_rows": int(len(audit_df)),
    "n_candidates_generated": int(candidates_df["quote_in_text"].notna().sum()),
    "n_validation_survivors": int((candidates_df["validation_verdict"] == "Yes").sum()),
    "n_pairs_with_overlap_flag": _overlap_pair_count,
}])
run_metadata_df.to_csv("phase10_assurance_run_metadata.csv", index=False)

print(
    f"Outputs written:\n"
    f"  pipeline_audit.csv               ({len(audit_df)} rows)\n"
    f"  phase10_assurance_per_theme_metrics.csv ({len(metrics_df)} rows)\n"
    f"  phase10_assurance_run_metadata.csv    (1 row)"
)


## Stakeholder report (Markdown + HTML)

Build a readable summary of the pipeline output for a non-technical audience.

Every block of text in the report is pulled directly from existing pipeline
outputs -- there are NO new LLM calls in this cell. Specifically:

- Theme names come from `themes_dict`.
- Quote text comes verbatim from `df["Free Words"]` via the audit (it was
  already a character-for-character copy at extraction time).
- Citizen labels come from `df["Citizen "]`.
- Per-theme counts are computed by groupby on `audit_df`.
- Sub-theme names and definitions come from `sub_themes_by_theme` (the
  Stage 6a output).

Outputs:
- `phase10_report.md` -- plain Markdown.
- `phase10_report.html` -- self-contained HTML with inline CSS so a
  stakeholder can double-click and view in any browser.


In [ ]:
# Stakeholder report -- pure string-formatting from existing pipeline outputs.
# No LLM calls in this cell.

# Per-pair lookup (one row per response x theme) for strict-AI-Yes counts.
_pair_key = audit_df["pair_id"].str.split("::").str[:2].str.join("::")
_pair_view = audit_df.assign(_pair_key=_pair_key).drop_duplicates(subset="_pair_key")


def _citizen_label(row_idx):
    """Pull the citizen identifier from df, or fall back to the row index."""
    val = df.at[row_idx, "Citizen "] if "Citizen " in df.columns else None
    if pd.isna(val) or str(val).strip() == "":
        return f"Response {row_idx}"
    return str(val).strip()


def _row_idx_from_pair_id(pair_id):
    """First segment of pair_id is always the row index."""
    return int(str(pair_id).split("::")[0])


_now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
_lines = []

# --- Header --------------------------------------------------------------
_lines.append("# Thematic Analysis Report")
_lines.append("")
_lines.append(f"*Generated {_now_utc}  ·  {len(df)} responses  ·  {len(themes_dict)} themes*")
_lines.append("")

# --- Summary table -------------------------------------------------------
_lines.append("## Summary")
_lines.append("")
_lines.append("| Theme | Responses raising it | Sub-themes |")
_lines.append("|---|---:|---:|")
for theme_name in themes_dict:
    n_yes = int(_pair_view[
        (_pair_view["Parent theme"] == theme_name) & _pair_view["AI Yes (strict)"]
    ].shape[0])
    n_sub = (
        len(sub_themes_by_theme[theme_name].sub_themes)
        if theme_name in sub_themes_by_theme else 0
    )
    _lines.append(f"| {theme_name} | {n_yes} | {n_sub} |")
_lines.append("")
_lines.append("---")
_lines.append("")

# --- Per theme ----------------------------------------------------------
for theme_name in themes_dict:
    n_yes = int(_pair_view[
        (_pair_view["Parent theme"] == theme_name) & _pair_view["AI Yes (strict)"]
    ].shape[0])

    _lines.append(f"## {theme_name}")
    _lines.append("")
    _lines.append(f"*{n_yes} of {len(df)} responses raised this theme.*")
    _lines.append("")

    # Sub-themes within this theme.
    if theme_name in sub_themes_by_theme:
        consolidated = sub_themes_by_theme[theme_name]
        # Count of quotes assigned per sub-theme on this parent.
        sub_counts = (
            audit_df[
                (audit_df["Parent theme"] == theme_name)
                & audit_df["Sub-theme"].notna()
                & (audit_df["Sub-theme"] != NONE_LABEL)
            ].groupby("Sub-theme").size().to_dict()
        )

        _lines.append("### Sub-themes")
        _lines.append("")
        for sub in consolidated.sub_themes:
            n_assigned = int(sub_counts.get(sub.name, 0))
            _lines.append(f"**{sub.name}**  *(count: {n_assigned})*")
            _lines.append("")
            _lines.append(f"*{sub.definition}*")
            _lines.append("")
            if n_assigned > 0:
                examples = audit_df[
                    (audit_df["Parent theme"] == theme_name)
                    & (audit_df["Sub-theme"] == sub.name)
                    & audit_df["Quote Text"].notna()
                ].head(3)
                for _, ex in examples.iterrows():
                    row_idx = _row_idx_from_pair_id(ex["pair_id"])
                    cit = _citizen_label(row_idx)
                    _lines.append(f"- \"{ex['Quote Text']}\"  — *{cit}*")
                _lines.append("")
    else:
        _lines.append("*No sub-themes were derived for this theme (no surviving quotes).*")
        _lines.append("")

    _lines.append("---")
    _lines.append("")

# Assemble + save Markdown.
report_md = "\n".join(_lines)
with open("phase10_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

# Convert to a self-contained HTML doc. Falls back gracefully if the
# `markdown` package isn't installed; the .md file always lands.
try:
    import markdown as _markdown_lib
    report_body_html = _markdown_lib.markdown(report_md, extensions=["tables"])
    html_doc = f"""<!doctype html>
<html lang=\"en\">
<head>
<meta charset=\"utf-8\">
<title>Thematic Analysis Report</title>
<style>
  body {{ font-family: Georgia, serif; max-width: 820px; margin: 2em auto; padding: 0 1em; line-height: 1.55; color: #1f2a1f; }}
  h1, h2, h3 {{ color: #2c5f2d; }}
  h1 {{ border-bottom: 2px solid #97bc62; padding-bottom: 0.3em; }}
  h2 {{ border-bottom: 1px solid #97bc62; padding-bottom: 0.2em; margin-top: 2.5em; }}
  blockquote {{ border-left: 4px solid #97bc62; margin: 0.5em 0; padding: 0.6em 1em; background: #f7f4ec; }}
  em {{ color: #556055; }}
  table {{ border-collapse: collapse; width: 100%; margin: 1em 0; }}
  th, td {{ border: 1px solid #97bc62; padding: 0.45em 0.85em; text-align: left; }}
  th {{ background: #e6eed8; color: #2c5f2d; }}
  hr {{ border: none; border-top: 1px dashed #97bc62; margin: 2em 0; }}
  ul {{ padding-left: 1.4em; }}
  li {{ margin-bottom: 0.3em; }}
</style>
</head>
<body>
{report_body_html}
</body>
</html>
"""
    with open("phase10_report.html", "w", encoding="utf-8") as f:
        f.write(html_doc)
    print(f"Wrote phase10_report.md ({len(report_md)} chars) and phase10_report.html")
except ImportError:
    print(
        f"Wrote phase10_report.md ({len(report_md)} chars).\n"
        f"To also generate phase10_report.html, install the markdown package:\n"
        f"  pip install markdown"
    )
